In [ ]:
from glob import glob
from tqdm import tqdm
import os
from os.path import join, basename
import re
import matplotlib.pyplot as plt
from collections import OrderedDict
import pandas as pd
import numpy as np
import argparse

from PIL import Image
import SimpleITK as sitk
import torch
import torch.multiprocessing as mp
from sam2.build_sam import build_sam2_video_predictor_npz
import SimpleITK as sitk
from skimage import measure, morphology
from pathlib import Path
import time


In [ ]:
torch.set_float32_matmul_precision('high')
torch.manual_seed(2024)
torch.cuda.manual_seed(2024)
np.random.seed(2024)

parser = argparse.ArgumentParser()


In [ ]:
import os

# Define your directory and output file
npz_dir = "/gpfs/home/machlm03/Segmentation/IWOAI_Segmentation_Challenge/test/npz/"
output_txt = "/gpfs/home/machlm03/Segmentation/IWOAI_Segmentation_Challenge/test/npz_file_list.txt"

# Get all .npz files
npz_files = sorted([f for f in os.listdir(npz_dir) if f.endswith(".npz")])

# Write to text file
with open(output_txt, "w") as f:
    for fname in npz_files:
        f.write(fname.split('.')[0] + "\n")

print(f"Saved {len(npz_files)} filenames to {output_txt}")


In [ ]:

parser = argparse.ArgumentParser()

parser.add_argument(
    '--checkpoint',
    type=str,
    default="checkpoints/MedSAM2_latest.pt",
    help='checkpoint path',
)
parser.add_argument(
    '--cfg',
    type=str,
    default="configs/sam2.1_hiera_t512.yaml",
    help='model config',
)

parser.add_argument(
    '--imgs_path',
    type=str,
    default="/gpfs/home/machlm03/Segmentation/OAI_demo/OAI_TrainTest/V00_00m_MultiClass/npz/",
    help='imgs path',
)
parser.add_argument(
    '--gts_path',
      type=str,
    default="/gpfs/home/machlm03/Segmentation/OAI_demo/OAI_TrainTest/V00_00m_MultiClass/npz/",
    help='simulate prompts based on ground truth',
)



parser.add_argument(
    '--test_files',
     type=str,
    default="/gpfs/home/machlm03/Segmentation/OAI_demo/OAI_TrainTest/cv_txt_files/test/test_split.txt",
    help='whether to propagate with box'
)




parser.add_argument(
    '--pred_save_dir',
    type=str,
    default="/gpfs/home/machlm03/Segmentation/OAI_demo/MedSam2_OAI_Inference_bbx_test/",
    help='path to save segmentation results',
)
# add option to propagate with either box or mask
parser.add_argument(
    '--propagate_with_box',
    default=True,
    action='store_true',
    help='whether to propagate with box'
)




parser.add_argument(
    '--method',
    type=str,
    choices=["percentiles", "max_classes"],  # restrict valid options
    default="max_classes",
    help='select method from percentile or max_classes',
)

parser.add_argument(
    "--percentile",
    type=int,
    nargs="+",
    default=[30],
    help="Percentiles to select prompt masks from (e.g., 25 50 75)"
)



In [ ]:
# args = parser.parse_args()
args, _ = parser.parse_known_args()
# args.checkpoint="/gpfs/home/machlm03/Segmentation/MedSAM2/medsam2.2_bbx_only_exp_log/fold5/checkpoints/checkpoint_75.pt"
args.checkpoint= "/gpfs/home/machlm03/Segmentation/MedSAM2/medsam2.2_exp_log/fold5/checkpoints/checkpoint_75.pt"
args.checkpoint="/gpfs/home/machlm03/Segmentation/MedSAM2/medsam2.3_MSAM_ecnoderfreeze/5/checkpoints/checkpoint_76.pt"



args.imgs_path = "/gpfs/home/machlm03/Segmentation/IWOAI_Segmentation_Challenge/test/npz/"
args.gts_path = "/gpfs/home/machlm03/Segmentation/IWOAI_Segmentation_Challenge/test/npz/"
args.pred_save_dir ='/gpfs/home/machlm03/Segmentation/IWOAI_Segmentation_Challenge/bbx_inference/'
args.test_files = "/gpfs/home/machlm03/Segmentation/IWOAI_Segmentation_Challenge/test/npz_file_list.txt"


checkpoint = args.checkpoint
model_cfg = args.cfg
imgs_path = args.imgs_path
gts_path = args.gts_path

pred_save_dir = args.pred_save_dir

# test_files='/gpfs/home/machlm03/Segmentation/OAI_demo/IWOAI_Segmentation_Challenge/test/npz_file_list.txt'
# test_files=args.test_files

test_files = args.test_files
# percentiles=args.percentiles
args.method="percentile"
method=args.method

os.makedirs(pred_save_dir, exist_ok=True)
propagate_with_box = args.propagate_with_box
percentiles=args.percentile


In [ ]:
# !split -l 300 /gpfs/home/machlm03/Segmentation/OAI_demo/OAI_TrainTest/cv_txt_files/test/test_split.txt /gpfs/home/machlm03/Segmentation/OAI_demo/OAI_TrainTest/cv_txt_files/test_chunk/chunk_
# !for f in /gpfs/home/machlm03/Segmentation/OAI_demo/OAI_TrainTest/cv_txt_files/test_chunk/chunk_*; do mv "$f" "$f.txt"; done
imgs_path

In [ ]:

def getLargestCC(segmentation):
    labels = measure.label(segmentation)
    largestCC = labels == np.argmax(np.bincount(labels.flat)[1:])+1

    # print(labels)
    return largestCC


def processMultiClassCC(segmentation):
    """
    Process connected components for multi-class segmentation.
    Preserves class labels while cleaning up components.
    """
    
    unique_classes = np.unique(segmentation)
    print(unique_classes)
    unique_classes = unique_classes[unique_classes > 0]  # Remove background
    
    print(f"Processing {len(unique_classes)} classes: {unique_classes}")
    
    processed_seg = np.zeros_like(segmentation)
    
    for class_id in unique_classes:
        # Extract this class only
        class_mask = (segmentation == class_id)
        
        if np.sum(class_mask) > 0:
            # Get largest connected component for this class
            class_labels = measure.label(class_mask)
            if class_labels.max() > 0:
                # Keep largest CC (or modify to keep all CCs)
                largest_cc = class_labels == np.argmax(np.bincount(class_labels.flat)[1:]) + 1
                processed_seg[largest_cc] = class_id
                
                print(f"Class {class_id}: {np.sum(largest_cc)} voxels")
    
    return processed_seg



def dice_multi_class(preds, targets):
    smooth = 1.0
    assert preds.shape == targets.shape
    labels = np.unique(targets)[1:]
    dices = []
    for label in labels:
        pred = preds == label
        target = targets == label
        intersection = (pred * target).sum()
        dices.append((2.0 * intersection + smooth) / (pred.sum() + target.sum() + smooth))
    return np.mean(dices)

def show_mask(mask, ax, mask_color=None, alpha=0.5):
    """
    show mask on the image

    Parameters
    ----------
    mask : numpy.ndarray
        mask of the image
    ax : matplotlib.axes.Axes
        axes to plot the mask
    mask_color : numpy.ndarray
        color of the mask
    alpha : float
        transparency of the mask
    """
    if mask_color is not None:
        color = np.concatenate([mask_color, np.array([alpha])], axis=0)
    else:
        color = np.array([251/255, 252/255, 30/255, alpha])
    h, w = mask.shape[-2:]
    mask_image = mask.reshape(h, w, 1) * color.reshape(1, 1, -1)
    ax.imshow(mask_image)


def show_box(box, ax, edgecolor='blue'):
    """
    show bounding box on the image

    Parameters
    ----------
    box : numpy.ndarray
        bounding box coordinates in the original image
    ax : matplotlib.axes.Axes
        axes to plot the bounding box
    edgecolor : str
        color of the bounding box
    """
    x0, y0 = box[0], box[1]
    w, h = box[2] - box[0], box[3] - box[1]
    ax.add_patch(plt.Rectangle((x0, y0), w, h, edgecolor=edgecolor, facecolor=(0,0,0,0), lw=2))     


def resize_grayscale_to_rgb_and_resize(array, image_size):
    """
    Resize a 3D grayscale NumPy array to an RGB image and then resize it.
    
    Parameters:
        array (np.ndarray): Input array of shape (d, h, w).
        image_size (int): Desired size for the width and height.
    
    Returns:
        np.ndarray: Resized array of shape (d, 3, image_size, image_size).
    """
    d, h, w = array.shape
    resized_array = np.zeros((d, 3, image_size, image_size))
    
    for i in range(d):
        img_pil = Image.fromarray(array[i].astype(np.uint8))
        img_rgb = img_pil.convert("RGB")
        img_resized = img_rgb.resize((image_size, image_size))
        img_array = np.array(img_resized).transpose(2, 0, 1)  # (3, image_size, image_size)
        resized_array[i] = img_array
    
    return resized_array

def mask2D_to_bbox(gt2D, max_shift=20):
    y_indices, x_indices = np.where(gt2D > 0)

        # Check if the mask has non-zero elements, i.e., a valid bounding box can be determined
    if len(x_indices) == 0 or len(y_indices) == 0:
        raise ValueError("Empty mask passed to mask2D_to_bbox; cannot determine bbox.")
        
    x_min, x_max = np.min(x_indices), np.max(x_indices)
    y_min, y_max = np.min(y_indices), np.max(y_indices)
    H, W = gt2D.shape
    bbox_shift = np.random.randint(0, max_shift + 1, 1)[0]
    x_min = max(0, x_min - bbox_shift)
    x_max = min(W-1, x_max + bbox_shift)
    y_min = max(0, y_min - bbox_shift)
    y_max = min(H-1, y_max + bbox_shift)
    boxes = np.array([x_min, y_min, x_max, y_max])
    return boxes

def mask3D_to_bbox(gt3D, max_shift=20):
    z_indices, y_indices, x_indices = np.where(gt3D > 0)
    x_min, x_max = np.min(x_indices), np.max(x_indices)
    y_min, y_max = np.min(y_indices), np.max(y_indices)
    z_min, z_max = np.min(z_indices), np.max(z_indices)
    D, H, W = gt3D.shape
    bbox_shift = np.random.randint(0, max_shift + 1, 1)[0]
    x_min = max(0, x_min - bbox_shift)
    x_max = min(W-1, x_max + bbox_shift)
    y_min = max(0, y_min - bbox_shift)
    y_max = min(H-1, y_max + bbox_shift)
    z_min = max(0, z_min)
    z_max = min(D-1, z_max)
    boxes3d = np.array([x_min, y_min, z_min, x_max, y_max, z_max])
    return boxes3d




def get_percentile_slices_from_npz(npz_path, method, percentiles, class_ids=None):
    data = np.load(npz_path)
    gt = data['gts']  # shape: [D, H, W]
    selected_slices = []
    indexs = []
    
    # Set default class_ids if not provided - moved to top
    if class_ids is None:
        class_ids = [c for c in np.unique(gt) if c > 0]  # Exclude background
    
    if method == 'percentile':
        if percentiles is None:
            raise ValueError("percentiles must be provided for 'percentile' method")
            
        # slice_foreground_counts = np.sum(gt > 0, axis=(1, 2))
        # valid_slices = np.where(slice_foreground_counts > 0)[0]
    
        # if len(valid_slices) == 0:
        #     raise ValueError(f"No foreground found in {npz_path}")
    
        # for p in percentiles:
        #     # Fix: use percentile of indices, not values
        #     percentile_idx = int(len(valid_slices) * p / 100)
        #     percentile_idx = min(percentile_idx, len(valid_slices) - 1)  # Ensure within bounds
        #     idx = valid_slices[percentile_idx]
        #     selected_slices.append(gt[idx])
        #     indexs.append(idx)
        # print(f"Selected slice {idx}")
        else:
            for idx in percentiles:
                # idx = valid_slices[p]
                selected_slices.append(gt[idx])
                indexs.append(idx)
                print(f"Selected slice {idx}")


            
    elif method == 'max_classes':
        # Select slice with maximum number of different classes
        D = gt.shape[0]
        print(f"Total slices: {D}")
        slice_scores = []
        for i in range(D):
            slice_data = gt[i]
            # Count number of different classes present
            present_classes = len([c for c in class_ids if np.sum(slice_data == c) > 0])
            slice_scores.append(present_classes)
        
        key_slice_idx = np.argmax(slice_scores)
        max_classes = slice_scores[key_slice_idx]
        selected_slices.append(gt[key_slice_idx])
        indexs.append(key_slice_idx)
        print(f"Selected slice {key_slice_idx} with {max_classes} classes")
        
    return selected_slices, indexs


In [ ]:
def dice_score(pred, gt):
    pred = pred > 0
    gt = gt > 0
    intersection = np.logical_and(pred, gt).sum()
    return 2.0 * intersection / (pred.sum() + gt.sum() + 1e-8)

def iou_score(pred, gt):
    pred = pred > 0
    gt = gt > 0
    intersection = np.logical_and(pred, gt).sum()
    union = np.logical_or(pred, gt).sum()
    return intersection / (union + 1e-8)

def mask_ids_to_rgb(mask_ids, class_colors):
    h, w = mask_ids.shape
    rgb = np.zeros((h, w, 3), dtype=np.uint8)
    for cls, color in class_colors.items():
        rgb[mask_ids == cls] = color
    return rgb



In [ ]:
# DL_info = pd.read_csv('CT_DeepLesion/DeepLesion_Dataset_Info.csv')

with open(test_files, "r") as f:
    allowed_cases = {line.strip() for line in f if line.strip()}
    allowed_cases = {case + ".npz" for case in allowed_cases}


npz_fnames = sorted(os.listdir(imgs_path))
npz_fnames = [i for i in npz_fnames if i.endswith('.npz')]
npz_fnames = [i for i in npz_fnames if not i.startswith('._')]
npz_fnames = [i for i in npz_fnames if i in allowed_cases]

print(f'Processing {len(npz_fnames)} nii files')
seg_info = OrderedDict()
seg_info['nii_name'] = []
seg_info['key_slice_index'] = []
seg_info['DICOM_windows'] = []
# initialized predictor
predictor = build_sam2_video_predictor_npz(model_cfg, checkpoint)


In [ ]:
with open(test_files, "r") as f:
    allowed_cases = {line.strip() for line in f if line.strip()}
    print(allowed_cases)
    allowed_cases = {case + ".npz" for case in allowed_cases}

'''uncoment for oai dataset'''
# npz_fnames = sorted(os.listdir(imgs_path))
# npz_fnames = [i for i in npz_fnames if i.endswith('.npz')]
# npz_fnames = [i for i in npz_fnames if not i.startswith('._')]
# npz_fnames = [i for i in npz_fnames if i in allowed_cases]

print(f'Processing {len(npz_fnames)} nii files')


In [ ]:
from tqdm import tqdm
import numpy as np
import torch
from os.path import join
allowed_cases = ["test_001_V00.npz"]
# allowed_cases=["9392241_00m_LEFT_SAG_3D_DESS_WE.npz"]
all_prompt_slices = []
# percentiles=[20]
percentiles_slices = 30,55
custom=[0,0,290,290]

In [ ]:

def processAllCC(segs_3D):
    """
    Process all connected components in 3D segmentation
    Each connected component gets a unique label starting from 1
    """
    if np.max(segs_3D) == 0:
        return segs_3D
    
    # Get all connected components
    labels = measure.label(segs_3D > 0)  # Only consider non-zero regions
    
    # Optional: filter out very small components
    min_size = 10  # adjust this threshold as needed
    for region in measure.regionprops(labels):
        if region.area < min_size:
            labels[labels == region.label] = 0
    
    # Relabel to ensure consecutive numbering
    labels = measure.label(labels > 0)
    
    return labels
    
def analyzeConnectedComponents(segs_3D):
    """
    Analyze all connected components in detail
    """
    # from skimage import measure
    
    labels = processAllCC(segs_3D)
    regions = measure.regionprops(labels)
    
    print(f"\nFound {len(regions)} connected components:")
    for i, region in enumerate(regions):
        print(f"Component {region.label}:")
        print(f"  - Volume: {region.area} voxels")
        print(f"  - Bounding box: {region.bbox}")
        print(f"  - Centroid: {region.centroid}")
        print(f"  - Equivalent diameter: {region.equivalent_diameter:.2f}")
    
    return labels, regions



for npz_file in tqdm(allowed_cases):
    # Load data
    data = np.load(join(imgs_path, npz_file))
    nii_image_data = data['imgs']  # shape: [D, H, W]
    gt_data = data['gts']          # shape: [D, H, W]

    segs_3D = np.zeros_like(nii_image_data, dtype=np.uint8)

    img_3D_ori = nii_image_data

    # Get key slices and indices
    slices, indices = get_percentile_slices_from_npz(join(imgs_path, npz_file), method='percentile',percentiles=percentiles_slices,class_ids=None)
    print(percentiles)
    


    # Resize and normalize the full volume
    img_resized = resize_grayscale_to_rgb_and_resize(img_3D_ori, 512)  # shape: [D, 3, 512, 512]
    img_resized = img_resized.astype(np.float32) / 255.0
    img_resized = torch.from_numpy(img_resized).cuda()




    img_mean=(0.485, 0.456, 0.406)
    img_std=(0.229, 0.224, 0.225)
    img_mean = torch.tensor(img_mean, dtype=torch.float32)[:, None, None].cuda()
    img_std = torch.tensor(img_std, dtype=torch.float32)[:, None, None].cuda()
    img_resized -= img_mean
    img_resized /= img_std
    z_mids = []





    slice_idx_start = 0
    start_time = time.time()


    for slice_2d, key_slice_idx in zip(slices, indices):
        key_slice_img = nii_image_data[key_slice_idx]
        key_slice_idx_offset = key_slice_idx - slice_idx_start
        slice_idx_start = key_slice_idx
        

        # bbox = mask2D_to_bbox(slice_2d)
            # bbox=[1,2,4,5]
            # print(bbox)
    
        prompt_slice = gt_data[int(key_slice_idx)]
        bbox = mask2D_to_bbox(prompt_slice)
        # bbox=[20,90,150,200]
        # print(bbox)
        # bbox=custom
        print(bbox)

    
        video_height, video_width = nii_image_data[0].shape
        slice_idx_start=0
        start_idx =0
    
    
        key_slice_idx_offset = key_slice_idx- slice_idx_start
    
    
    
            
    
        with torch.inference_mode(), torch.autocast("cuda", dtype=torch.bfloat16):
            inference_state = predictor.init_state(img_resized, video_height, video_width)
            if propagate_with_box:
                _, out_obj_ids, out_mask_logits = predictor.add_new_points_or_box(
                                                    inference_state=inference_state,
                                                    frame_idx=key_slice_idx_offset,
                                                    obj_id=1,
                                                    box=bbox,
                                                )
            else: # gt
                pass
    
            for out_frame_idx, out_obj_ids, out_mask_logits in predictor.propagate_in_video(inference_state):
                segs_3D[out_frame_idx, (out_mask_logits[0] > 0.0).cpu().numpy()[0]] = 1

            # predictor.reset_state(inference_state)

            if propagate_with_box:
                _, out_obj_ids, out_mask_logits = predictor.add_new_points_or_box(
                                                    inference_state=inference_state,
                                                    frame_idx=key_slice_idx_offset,
                                                    obj_id=1,
                                                    box=bbox,
                                                )
            else: # gt
                pass
    
            for out_frame_idx, out_obj_ids, out_mask_logits in predictor.propagate_in_video(inference_state, reverse=True):
                segs_3D[out_frame_idx, (out_mask_logits[0] > 0.0).cpu().numpy()[0]] = 1
            predictor.reset_state(inference_state)
            


    # Use it in your main loop:
    if np.max(segs_3D) > 0:
        segs_3D_labeled, component_stats = analyzeConnectedComponents(segs_3D)
        segs_3D = segs_3D_labeled.astype(np.uint8)
    
    np.savez_compressed(
        os.path.join(pred_save_dir, npz_file),
        imgs=img_3D_ori.astype(np.uint8),  # original image
        pre=segs_3D.astype(np.uint8),     # predicted mask
        gts=gt_data.astype(np.uint8) 
    )

    



In [ ]:
from tqdm import tqdm
import numpy as np
import torch
from os.path import join
custom=[40, 0, 290, 290] 
# allowed_cases = ["test_001_V00.npz"]

npz_files = allowed_cases
all_prompt_slices = []
percentiles = [33,50,75]

# Alternative approach: Use different prompts for different classes
# This could be different bounding boxes, points, or masks for each class
def get_class_specific_prompts(gt_slice, bbox, num_classes=4):
    """
    Generate different prompts for different classes.
    This is a placeholder - you'll need to implement based on your specific needs.
    """
    prompts = []
    
    # Method 1: Divide bounding box into regions
    x1, y1, x2, y2 = bbox
    width = x2 - x1
    height = y2 - y1
    

    
    # Method 2: If you have ground truth masks for different classes
    unique_labels = np.unique(gt_slice)
    for label in unique_labels[1:]:  # Skip background
        class_mask = (gt_slice == label)
        if np.any(class_mask):
            class_bbox = mask2D_to_bbox(class_mask)
            prompts.append(('box', class_bbox))
    
    return prompts


for npz_file in tqdm(npz_files):
    # Load data
    data = np.load(join(imgs_path, npz_file))
    nii_image_data = data['imgs']  # shape: [D, H, W]
    gt_data = data['gts']          # shape: [D, H, W]
    
    # Check what classes exist in ground truth
    gt_unique_classes = np.unique(gt_data)
    print(f"Ground truth classes: {gt_unique_classes}")
    num_classes = len(gt_unique_classes) - 1 if 0 in gt_unique_classes else len(gt_unique_classes)
    print(f"Number of non-background classes: {num_classes}")
    
    # Initialize multi-class segmentation array
    segs_3D = np.zeros_like(nii_image_data, dtype=np.uint8)
    img_3D_ori = nii_image_data
    
    # Get key slices and indices
    # slices, indices = get_percentile_slices_from_npz(join(imgs_path, npz_file), percentiles)
    slices, indices = get_percentile_slices_from_npz(join(imgs_path, npz_file),method='percentile',percentiles=percentiles_slices,class_ids=None)
    print(indices)

    
    # Calculate total iterations for progress bar
    total_iterations = len(slices) * num_classes
    
    # Resize and normalize the full volume
    img_resized = resize_grayscale_to_rgb_and_resize(img_3D_ori, 512)
    img_resized = img_resized.astype(np.float32) / 255.0
    img_resized = torch.from_numpy(img_resized).cuda()
    img_mean = (0.485, 0.456, 0.406)
    img_std = (0.229, 0.224, 0.225)
    img_mean = torch.tensor(img_mean, dtype=torch.float32)[:, None, None].cuda()
    img_std = torch.tensor(img_std, dtype=torch.float32)[:, None, None].cuda()
    img_resized -= img_mean
    img_resized /= img_std
    
    start_time = time.time()

    
    # Single progress bar for all slices and classes
    with tqdm(total=total_iterations, desc=f"Processing {npz_file}") as pbar:
        for slice_idx, (slice_2d, key_slice_idx) in enumerate(zip(slices, indices)):
            prompt_slice = gt_data[int(key_slice_idx)]
            original_bbox = mask2D_to_bbox(prompt_slice)
            # original_bbox = [20, 90, 150, 200]  # Your hardcoded bbox
            
            video_height, video_width = nii_image_data[0].shape
            key_slice_idx_offset = key_slice_idx
            
            # Get class-specific prompts
            class_prompts = get_class_specific_prompts(prompt_slice, original_bbox, num_classes)
            
            with torch.inference_mode(), torch.autocast("cuda", dtype=torch.bfloat16):
                # Process each class with its specific prompt
                for class_id, (prompt_type, prompt_data) in enumerate(class_prompts, 1):
                    # Update progress bar description with current status
                    pbar.set_description(f"Processing {npz_file} - Slice {slice_idx+1}/{len(slices)}, Class {class_id}/{num_classes}")
                    
                    # Create separate segmentation for this class
                    class_segs_3D = np.zeros_like(nii_image_data, dtype=np.uint8)
                    
                    # Initialize inference state for this class
                    inference_state = predictor.init_state(img_resized, video_height, video_width)
                    
                    if propagate_with_box and prompt_type == 'box':
                        _, out_obj_ids, out_mask_logits = predictor.add_new_points_or_box(
                            inference_state=inference_state,
                            frame_idx=key_slice_idx_offset,
                            obj_id=1,  # Always use obj_id=1, but different inference_state
                            box=prompt_data,
                        )
                        
                        # Forward propagation
                        for out_frame_idx, out_obj_ids, out_mask_logits in predictor.propagate_in_video(inference_state):
                            mask = (out_mask_logits[0] > 0.0).cpu().numpy()[0]
                            class_segs_3D[out_frame_idx, mask] = 1
                        
                        predictor.reset_state(inference_state)
                        
                        # Backward propagation
                        inference_state = predictor.init_state(img_resized, video_height, video_width)
                        _, out_obj_ids, out_mask_logits = predictor.add_new_points_or_box(
                            inference_state=inference_state,
                            frame_idx=key_slice_idx_offset,
                            obj_id=1,
                            box=prompt_data,
                        )
                        
                        for out_frame_idx, out_obj_ids, out_mask_logits in predictor.propagate_in_video(inference_state, reverse=True):
                            mask = (out_mask_logits[0] > 0.0).cpu().numpy()[0]
                            class_segs_3D[out_frame_idx, mask] = 1
                        
                        predictor.reset_state(inference_state)
                    
                    # Apply largest connected component to this class
                    if np.max(class_segs_3D) > 0:
                        class_segs_3D = getLargestCC(class_segs_3D)
                        class_segs_3D = np.uint8(class_segs_3D)
                        
                        # Assign this class to final segmentation
                        # Only assign to background pixels to avoid conflicts
                        background_mask = (segs_3D == 0)
                        class_mask = (class_segs_3D > 0)
                        final_assignment = background_mask & class_mask
                        segs_3D[final_assignment] = class_id
                        
                        # Update progress bar postfix with current results
                        assigned_pixels = np.sum(final_assignment)
                        pbar.set_postfix({
                            'assigned_pixels': assigned_pixels,
                            'current_classes': len(np.unique(segs_3D)) - 1  # -1 for background
                        })
                    else:
                        pbar.set_postfix({
                            'assigned_pixels': 0,
                            'current_classes': len(np.unique(segs_3D)) - 1
                        })
                    
                    # Update progress bar
                    pbar.update(1)
    
    print(f"Final unique classes: {np.unique(segs_3D)}")
    
    # Save results
    np.savez_compressed(
        os.path.join(pred_save_dir, npz_file),
        imgs=img_3D_ori.astype(np.uint8),
        pre=segs_3D.astype(np.uint8),
        gts=gt_data.astype(np.uint8)
    )
    
    end_time = time.time()
    duration = end_time - start_time
    print(f'Finished {npz_file} in {duration:.2f} seconds')
    print(f'Final predicted classes: {np.unique(segs_3D)}')

In [ ]:
import subprocess

# Run 'gcc --version' and capture the output
gcc_version = subprocess.run(["gcc", "--version"], capture_output=True, text=True)

# Print the first line of the output
print(gcc_version.stdout.splitlines()[0])


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import cv2
from pathlib import Path
percentile=percentiles[0]
# npz_files=['9002411_00m_LEFT_SAG_3D_DESS_WE.npz','9869252_00m_RIGHT_SAG_3D_DESS_WE.npz',"9870569_00m_LEFT_SAG_3D_DESS_WE.npz",'9871608_00m_RIGHT_SAG_3D_DESS_WE.npz','9873102_00m_LEFT_SAG_3D_DESS_WE.npz']
# # Define class colors for visualization
# npz_files = ["9013941_00m_LEFT_SAG_3D_DESS_WE.npz"]
# npz_files = ["9874238_00m_LEFT_SAG_3D_DESS_WE.npz"]

CLASS_COLORS = {
    0: (0, 0, 0),         # background
    1: (255, 0, 0),       # red
    2: (0, 255, 0),       # green
    3: (0, 0, 255),       # blue
    4: (255, 255, 0),     # yellow
    # 5: (255, 0, 255),     # magenta
    # 6: (0, 255, 255),     # cyan
    # 7: (255, 165, 0),     # orange ← updated
}


def build_color_lut(class_colors: dict) -> np.ndarray:
    lut = np.zeros((256, 3), dtype=np.uint8)
    for k, rgb in class_colors.items():
        lut[int(k)] = np.array(rgb, dtype=np.uint8)
    return lut

COLOR_LUT = build_color_lut(CLASS_COLORS)

def mask2D_to_bbox(mask_2d):
    """Extract bounding box from 2D mask"""
    if np.sum(mask_2d) == 0:
        return None
    rows, cols = np.where(mask_2d > 0)
    y_min, y_max = rows.min(), rows.max()
    x_min, x_max = cols.min(), cols.max()
    return [x_min, y_min, x_max, y_max]


def get_key_slice_index(gt_3d, percentile):
    """
    Get the key slice index based on percentile of non-zero voxels
    This mimics the get_percentile_slices_from_npz function behavior
    """
    # Calculate the amount of annotation per slice
    slice_annotations = [np.sum(gt_3d[i] > 0) for i in range(gt_3d.shape[0])]
    
    # Find slices with annotations
    annotated_slices = [i for i, count in enumerate(slice_annotations) if count > 0]
    
    if len(annotated_slices) == 0:
        return gt_3d.shape[0] // 2  # fallback to middle slice
    
    # Get percentile-based key slice
    percentile_idx = int(len(annotated_slices) * percentile / 100)
    key_slice_idx = annotated_slices[min(percentile_idx, len(annotated_slices) - 1)]
    
    return key_slice_idx


def get_class_bboxes_from_key_slice(gt_3d, class_ids, percentile):
    """
    Extract bounding boxes for each class from the key slice only
    
    Args:
        gt_3d: 3D numpy array with class labels [D, H, W]
        class_ids: list of class IDs to extract bboxes for
        percentile: percentile to determine key slice
    
    Returns:
        tuple: (key_slice_idx, bboxes_dict) where bboxes_dict is {class_id: bbox}
    """
    key_slice_idx = get_key_slice_index(gt_3d, percentile)
    key_slice_idx=69
    key_slice = gt_3d[key_slice_idx]
    
    print(f"Key slice index: {key_slice_idx}")
    print(f"Key slice unique classes: {np.unique(key_slice)}")
    
    bboxes = {}
    for class_id in class_ids:
        class_mask = (key_slice == class_id)
        bbox = mask2D_to_bbox(class_mask)
        # bbox = custom  # Your hardcoded bbox

        bboxes[class_id] = bbox

        # if bbox is not None:
        #     print(f"Class {class_id} bbox: {bbox}")
        # else:
        #     print(f"Class {class_id}: not present in key slice")
    
    return key_slice_idx, bboxes



def draw_multiple_bboxes(img_rgb, bboxes_dict, class_colors, thickness=2):
    """Draw multiple bounding boxes with class labels"""
    img_with_boxes = img_rgb.copy()
    
    for class_id, bbox in bboxes_dict.items():
        if bbox is not None:
            x_min, y_min, x_max, y_max = bbox           

            color = class_colors.get(class_id, (255, 255, 255))
            cv2.rectangle(img_with_boxes, (x_min, y_min), (x_max, y_max), color, thickness)
            
            # Add class label
            label_text = f"Class {class_id}"
            font_scale = 0.5
            font_thickness = 1
            (text_width, text_height), baseline = cv2.getTextSize(label_text, cv2.FONT_HERSHEY_SIMPLEX, font_scale, font_thickness)
            
            # Draw text background
            cv2.rectangle(img_with_boxes, 
                         (x_min, y_min - text_height - 5),
                         (x_min + text_width, y_min),
                         color, -1)
            
            # Draw text
            cv2.putText(img_with_boxes, label_text, 
                       (x_min, y_min - 2), 
                       cv2.FONT_HERSHEY_SIMPLEX, font_scale, (255, 255, 255), font_thickness)
    
    return img_with_boxes

def mask_ids_to_rgb(mask_ids, class_colors):
    h, w = mask_ids.shape
    rgb = np.zeros((h, w, 3), dtype=np.uint8)
    for cls, color in class_colors.items():
        rgb[mask_ids == cls] = color
    return rgb

def enhance_overlay(img_rgb, seg_rgb, alpha=0.6, beta=0.4):
    gray_seg = cv2.cvtColor(seg_rgb, cv2.COLOR_RGB2GRAY)
    edges = cv2.Canny(gray_seg, threshold1=50, threshold2=150)
    edge_highlight = cv2.cvtColor(edges, cv2.COLOR_GRAY2RGB)
    img_with_edges = cv2.addWeighted(img_rgb, 1, edge_highlight, 1, 0)
    overlay = cv2.addWeighted(img_with_edges, alpha, seg_rgb, beta, 0)
    return overlay

def dice_score(pred, gt):
    pred = pred > 0
    gt = gt > 0
    intersection = np.logical_and(pred, gt).sum()
    return 2.0 * intersection / (pred.sum() + gt.sum() + 1e-8)

def iou_score(pred, gt):
    pred = pred > 0
    gt = gt > 0
    intersection = np.logical_and(pred, gt).sum()
    union = np.logical_or(pred, gt).sum()
    return intersection / (union + 1e-8)
    
from scipy.spatial.distance import directed_hausdorff
import numpy as np

def hd95_score(pred, gt):
    pred = pred > 0
    gt = gt > 0

    pred_points = np.argwhere(pred)
    gt_points = np.argwhere(gt)

    if pred_points.size == 0 or gt_points.size == 0:
        return np.nan  # or some fallback value

    forward_hd = directed_hausdorff(pred_points, gt_points)[0]
    backward_hd = directed_hausdorff(gt_points, pred_points)[0]

    return np.percentile([forward_hd, backward_hd], 95)

def labels_to_rgb(label2d: np.ndarray) -> np.ndarray:
    return COLOR_LUT[label2d.astype(np.uint8)]

# === Load and visualize ===
# pred_save_dir="/gpfs/home/machlm03/Segmentation/OAI_demo/MedSam2_Finetune_OAI_Inference_bbx/"
data = np.load(os.path.join(pred_save_dir, npz_files[-1]))
img_3D = data['imgs']  # shape: [D, H, W]
gt_3D = data['gts']
pred_3D = data['pre']

# Define which classes to analyze
labels = [1,2,3,4,5, 6, 7]

# print("=== Key Slice Analysis ===")
# Get key slice bounding boxes (this matches your SAM2 approach)
key_slice_idx, gt_key_bboxes = get_class_bboxes_from_key_slice(gt_3D, labels, percentile)

# print("\n=== Representative Slice Analysis (for comparison) ===")
# Get representative bboxes (best slice for each class)
# gt_repr_bboxes, best_slice_info = get_representative_bboxes_3d(gt_3D, labels)

# Use the key slice for visualization (matching your SAM2 processing)
slice_idx = 69
img_slice = img_3D[slice_idx]
gt_slice = gt_3D[slice_idx]
pred_slice = pred_3D[slice_idx]

print(f"\nVisualizing slice {slice_idx} (key slice)")
print(f"GT classes in this slice: {np.unique(gt_slice)}")
print(f"Pred classes in this slice: {np.unique(pred_slice)}")

# Get prediction bboxes from the same key slice
pred_key_bboxes = {}
for class_id in labels:
    class_mask = (pred_slice == class_id)
    bbox = mask2D_to_bbox(class_mask)
    pred_key_bboxes[class_id] = bbox


# Contrast stretching
vmin, vmax = np.percentile(img_slice, [1, 99])
img_norm = np.clip((img_slice - vmin) / (vmax - vmin + 1e-8), 0, 1)
img_rgb = (np.stack([img_norm]*3, axis=-1) * 255).astype(np.uint8)

# Convert masks to RGB
gt_rgb = mask_ids_to_rgb(gt_slice, CLASS_COLORS)
pred_rgb = mask_ids_to_rgb(pred_slice, CLASS_COLORS)

# Create overlay
overlay = enhance_overlay(img_rgb, pred_rgb)

# Draw bounding boxes from KEY SLICE only
img_with_gt_key_boxes = draw_multiple_bboxes(img_rgb, gt_key_bboxes, CLASS_COLORS)
# img_with_pred_key_boxes = draw_multiple_bboxes(img_rgb, pred_key_bboxes, CLASS_COLORS)

# Draw representative bboxes for comparison (these might be from different slices)
# img_with_gt_repr_boxes = draw_multiple_bboxes(img_rgb, gt_repr_bboxes, CLASS_COLORS)

# Visualization
plt.figure(figsize=(30, 6))


plt.subplot(1, 4, 1)
plt.imshow(img_with_gt_key_boxes)
plt.title(f"GT Key Slice Bboxes\n(Slice {slice_idx})")
plt.axis('off')


plt.subplot(1, 4, 2)
plt.imshow(gt_rgb)
plt.title("Ground Truth Masks")
plt.axis('off')


plt.subplot(1, 4, 3)
plt.imshow(pred_rgb)
plt.title("Prediction Masks")
plt.axis('off')

plt.subplot(1, 4, 4)
plt.imshow(overlay)
plt.title("Overlay (Edge Enhanced)")
plt.axis('off')

plt.tight_layout()
plt.show()

# Evaluation
gt_bool = np.isin(gt_3D, labels)
pred_bool = np.isin(pred_3D, labels)
dice = dice_score(pred_bool, gt_bool)
iou = iou_score(pred_bool, gt_bool)

print(f"\n=== Evaluation Metrics ===")
print(f"Dice Score: {dice:.4f}")
print(f"IoU Score:  {iou:.4f}")

# Per-class evaluation
print(f"\n=== Overall Per-Class Evaluation for whole Voxel ===")
for class_id in labels:
    gt_class = (gt_3D == class_id)
    pred_class = (pred_3D == class_id)
    
    if np.sum(gt_class) > 0:
        class_dice = dice_score(pred_class, gt_class)
        class_iou = iou_score(pred_class, gt_class)
        print(f"Class {class_id} - Dice: {class_dice:.4f}, IoU: {class_iou:.4f}")
    else:
        print(f"Class {class_id} - Not present in ground truth")

print(f"\n=== Summary ===")
print(f"Key slice used for SAM2 processing: {key_slice_idx}")
print(f"Classes present in key slice: {[c for c, bbox in gt_key_bboxes.items() if bbox is not None]}")
print(f"Classes missing from key slice: {[c for c, bbox in gt_key_bboxes.items() if bbox is None]}")

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Patch
import cv2

CLASS_COLORS = {
    0: (0, 0, 0),       # background - black
    1: (255, 255, 0),   # class 1 - yellow
    2: (255, 0, 255),   # class 2 - magenta
    3: (0, 255, 255),   # class 3 - cyan
    4: (0, 255, 0),     # class 4 - green
}

CLASS_NAMES = {
    1: "Femur-Cart",
    2: "Tibia-Cart", 
    3: "Patella-Cart",
    4: "Meniscus",
}

def mask2D_to_bbox(mask_2d):
    """Extract bounding box from 2D mask"""
    if np.sum(mask_2d) == 0:
        return None
    rows, cols = np.where(mask_2d > 0)
    y_min, y_max = rows.min(), rows.max()
    x_min, x_max = cols.min(), cols.max()
    return [x_min, y_min, x_max, y_max]

def get_class_bboxes_2d(mask_slice, class_ids):
    """
    Extract bounding boxes for each class in a 2D slice
    
    Args:
        mask_slice: 2D numpy array with class labels
        class_ids: list of class IDs to extract bboxes for
    
    Returns:
        dict: {class_id: bbox} where bbox is [x_min, y_min, x_max, y_max] or None if class not present
    """
    bboxes = {}
    for class_id in class_ids:
        class_mask = (mask_slice == class_id)
        bbox = mask2D_to_bbox(class_mask)
        bboxes[class_id] = bbox
    return bboxes

def draw_multiple_bboxes(img, bboxes_dict, class_colors, thickness=2):
    """
    Draw multiple bounding boxes on an image
    
    Args:
        img: Image array (H, W, 3)
        bboxes_dict: Dictionary {class_id: bbox} where bbox is [x_min, y_min, x_max, y_max]
        class_colors: Dictionary mapping class_id to RGB colors
        thickness: Line thickness for bounding boxes
    
    Returns:
        Image with bounding boxes drawn
    """
    img_with_boxes = img.copy()
    
    for class_id, bbox in bboxes_dict.items():
        if bbox is not None:
            x_min, y_min, x_max, y_max = bbox
            color = class_colors.get(class_id, (255, 255, 255))  # Default to white if color not found
            
            # Draw rectangle
            cv2.rectangle(img_with_boxes, (x_min, y_min), (x_max, y_max), color, thickness)
            
            # Add class label
            label = CLASS_NAMES.get(class_id, f"C{class_id}")
            font_scale = 0.5
            font_thickness = 1
            (text_width, text_height), _ = cv2.getTextSize(label, cv2.FONT_HERSHEY_SIMPLEX, font_scale, font_thickness)
            
            # Draw text background
            cv2.rectangle(img_with_boxes, (x_min, y_min - text_height - 5), 
                         (x_min + text_width, y_min), color, -1)
            
            # Draw text
            cv2.putText(img_with_boxes, label, (x_min, y_min - 5), 
                       cv2.FONT_HERSHEY_SIMPLEX, font_scale, (0, 0, 0), font_thickness)
    
    return img_with_boxes

def enhance_contrast_overlay(base_img, mask_rgb, alpha=0.7, brightness_boost=1.5):
    """
    Create high-contrast overlay with enhanced visibility
    
    Args:
        base_img: Base image (H, W, 3) in range [0, 255]
        mask_rgb: Mask RGB image (H, W, 3) in range [0, 255]
        alpha: Opacity of mask overlay (higher = more mask visible)
        brightness_boost: Multiplier for mask brightness
    """
    # Convert to float for processing
    base_float = base_img.astype(np.float32) / 255.0
    mask_float = mask_rgb.astype(np.float32) / 255.0
    
    # Enhance mask brightness and contrast
    mask_enhanced = np.clip(mask_float * brightness_boost, 0, 1)
    
    # Apply stronger alpha blending
    overlay = (1 - alpha) * base_float + alpha * mask_enhanced
    
    # Additional contrast enhancement for non-background areas
    mask_binary = np.any(mask_enhanced > 0, axis=2, keepdims=True)
    overlay = np.where(mask_binary, 
                      np.clip(overlay * 1.2, 0, 1),  # Boost contrast where mask exists
                      base_float * 0.7)  # Dim background slightly
    
    return (overlay * 255).astype(np.uint8)

def dice_score(pred_bool, gt_bool):
    """Calculate Dice score for boolean masks"""
    intersection = np.logical_and(pred_bool, gt_bool).sum()
    total = pred_bool.sum() + gt_bool.sum()
    
    if total == 0:
        return 1.0 if intersection == 0 else 0.0
    
    return 2.0 * intersection / total

def visualize_percentile_slices(img_3D, gt_3D, pred_3D, labels, percentiles=None, 
                               bbox_percentiles=None, key_slice_idx=None, CLASS_COLORS=None, 
                               figsize_per_image=(4, 3)):
    """
    Visualize GT and prediction overlays in two columns with optional bounding boxes
    
    Args:
        img_3D: 3D image array [D, H, W]
        gt_3D: 3D ground truth array [D, H, W]
        pred_3D: 3D prediction array [D, H, W]
        labels: list of class labels to analyze
        percentiles: List of percentiles to visualize
        bbox_percentiles: List of percentiles where bounding boxes should be shown
        key_slice_idx: Index of reference slice for bounding box extraction
        CLASS_COLORS: Dictionary mapping class_id to RGB colors
        figsize_per_image: Size of each image (width, height)
    
    Returns:
        dict: Slice metrics including per-class Dice scores
    """
    
    # Default percentiles if not provided
    if percentiles is None:
        percentiles = list(range(0, 101, 10))
    
    # Default bbox percentiles (empty list means no bboxes)
    if bbox_percentiles is None:
        bbox_percentiles = []
    
    # Define default class colors if not provided
    if CLASS_COLORS is None:
        CLASS_COLORS = {
            0: (0, 0, 0),       # background - black
            1: (255, 255, 0),   # class 1 - yellow
            2: (255, 0, 255),   # class 2 - magenta
            3: (0, 255, 255),   # class 3 - cyan
            4: (0, 255, 0),     # class 4 - green
        }
    
    D = img_3D.shape[0]
    n_percentiles = len(percentiles)
    
    # Calculate slice indices for each percentile
    slice_indices = []
    for p in percentiles:
        slice_idx = int(D * p / 100)
        slice_indices.append(min(slice_idx, D-1))
    
    # Get reference slice for bounding boxes
    key_slice = None
    if key_slice_idx is not None and len(bbox_percentiles) > 0:
        key_slice = gt_3D[key_slice_idx]
    
    # Create figure with 2 columns: GT | Pred
    n_rows = n_percentiles
    n_cols = 2
    
    fig, axes = plt.subplots(n_rows, n_cols, 
                            figsize=(figsize_per_image[0] * n_cols, 
                                   figsize_per_image[1] * n_rows))
    
    # Handle single row case
    if n_rows == 1:
        axes = axes.reshape(1, -1)
    
    # Store metrics
    slice_metrics = {}
    all_dice_scores = {label: [] for label in labels}
    
    for row, (percentile, slice_idx) in enumerate(zip(percentiles, slice_indices)):
        img_slice = img_3D[slice_idx]
        gt_slice = gt_3D[slice_idx]
        pred_slice = pred_3D[slice_idx]
        
        # Normalize base image with better contrast
        vmin, vmax = np.percentile(img_slice, [2, 98])
        img_norm = np.clip((img_slice - vmin) / (vmax - vmin + 1e-8), 0, 1)
        
        # Apply gamma correction for better visibility
        img_norm = np.power(img_norm, 0.8)
        img_rgb = (np.stack([img_norm] * 3, axis=-1) * 255).astype(np.uint8)
        
        # Convert masks to RGB
        gt_rgb = mask_ids_to_rgb(gt_slice, CLASS_COLORS)
        pred_rgb = mask_ids_to_rgb(pred_slice, CLASS_COLORS)
        
        # Create high-contrast overlays
        gt_overlay = enhance_contrast_overlay(img_rgb, gt_rgb, alpha=0.6, brightness_boost=1.8)
        pred_overlay = enhance_contrast_overlay(img_rgb, pred_rgb, alpha=0.6, brightness_boost=1.8)
        
        # Check if this percentile should show bounding boxes
        show_bbox = percentile in bbox_percentiles and key_slice is not None
        
        if show_bbox:
            # Get bounding boxes from reference slice
            gt_bboxes = get_class_bboxes_2d(key_slice, labels)
            pred_bboxes = get_class_bboxes_2d(pred_slice, labels)
            
            # Draw bounding boxes on overlays
            gt_overlay_with_boxes = draw_multiple_bboxes(gt_overlay, gt_bboxes, CLASS_COLORS, thickness=2)
            pred_overlay_with_boxes = draw_multiple_bboxes(pred_overlay, pred_bboxes, CLASS_COLORS, thickness=2)
        else:
            gt_overlay_with_boxes = gt_overlay
            pred_overlay_with_boxes = pred_overlay
        
        # Calculate metrics for each class
        class_dice_scores = {}
        class_present_in_gt = []
        class_present_in_pred = []
        
        for class_id in labels:
            slice_gt_bool = (gt_slice == class_id)
            slice_pred_bool = (pred_slice == class_id)
            
            if np.sum(slice_gt_bool) > 0:
                class_present_in_gt.append(class_id)
            if np.sum(slice_pred_bool) > 0:
                class_present_in_pred.append(class_id)
            
            dice = dice_score(slice_pred_bool, slice_gt_bool)
            class_dice_scores[class_id] = dice
            all_dice_scores[class_id].append(dice)
        
        # Create legend for classes present
        legend_text = []
        legend_colors = []
        for class_id in labels:
            if class_id in class_present_in_gt or class_id in class_present_in_pred:
                color = tuple(c/255.0 for c in CLASS_COLORS[class_id])
                dice_val = class_dice_scores[class_id]
                class_name = CLASS_NAMES.get(class_id, f"C{class_id}")
                legend_text.append(f"{class_name}: {dice_val:.2f}")
                legend_colors.append(color)
        
        # Calculate overall dice for this slice
        valid_dice_scores = [dice for dice in class_dice_scores.values() if dice is not None]
        overall_dice = np.mean(valid_dice_scores) if valid_dice_scores else 0.0
        
        # Column 0: GT overlay
        gt_title = f'GT - {percentile}% (#{slice_idx})'
        if show_bbox:
            gt_title += ' [BBox]'
        gt_title += f'\nClasses: {class_present_in_gt}'
        
        axes[row, 0].imshow(gt_overlay_with_boxes)
        axes[row, 0].set_title(gt_title, fontsize=9, pad=3)
        axes[row, 0].axis('off')
        

        
        # Column 1: Prediction overlay
        pred_title = f'Pred - {percentile}%'
        if show_bbox:
            pred_title += ' [BBox]'
        pred_title += f'\nOverall Dice: {overall_dice:.3f}'
        
        axes[row, 1].imshow(pred_overlay_with_boxes)
        axes[row, 1].set_title(pred_title, fontsize=9, pad=3)
        axes[row, 1].axis('off')
        
        # Add legend to Pred
        if legend_text:
            legend_elements = [Patch(facecolor=color, edgecolor='black', linewidth=0.3) 
                             for color in legend_colors]
            axes[row, 1].legend(legend_elements, legend_text,
                               loc='upper right', bbox_to_anchor=(1, 1), 
                               fontsize=7, framealpha=0.9, handlelength=0.8,
                               handletextpad=0.3, borderaxespad=0.1)
        
        # Store metrics
        slice_metrics[percentile] = {
            'slice_idx': slice_idx,
            'dice_per_class': class_dice_scores,
            'gt_classes': class_present_in_gt,
            'pred_classes': class_present_in_pred,
            'overall_dice': overall_dice,
            'has_bbox': show_bbox
        }
    
    # Add column headers
    bbox_info = f" (BBox on: {bbox_percentiles})" if bbox_percentiles else ""
    fig.text(0.25, 0.98, f'Ground Truth{bbox_info}', fontsize=12, fontweight='bold', 
             ha='center', va='top', transform=fig.transFigure)
    fig.text(0.75, 0.98, f'Predictions{bbox_info}', fontsize=12, fontweight='bold', 
             ha='center', va='top', transform=fig.transFigure)
    

    plt.tight_layout()
    plt.subplots_adjust(top=0.91)
    plt.show()
    
    # Print summary statistics
    print("\n" + "="*70)
    print("DICE SCORE SUMMARY")
    print("="*70)
    
    if bbox_percentiles:
        print(f"Bounding boxes shown for percentiles: {bbox_percentiles}")
        if key_slice_idx is not None:
            print(f"Reference slice for bounding boxes: #{key_slice_idx}")
        print("-" * 70)
    
    # Per-class summary
    for label in labels:
        if all_dice_scores[label]:
            scores = [s for s in all_dice_scores[label] if s is not None]
            if scores:
                mean_dice = np.mean(scores)
                std_dice = np.std(scores)
                min_dice = np.min(scores)
                max_dice = np.max(scores)
                class_name = CLASS_NAMES.get(label, f"Class {label}")
                print(f"{class_name:15} | Mean: {mean_dice:.3f} ± {std_dice:.3f} | "
                      f"Range: [{min_dice:.3f}, {max_dice:.3f}] | Count: {len(scores)}")
    
    # Overall statistics
    all_scores = []
    for scores_list in all_dice_scores.values():
        all_scores.extend([s for s in scores_list if s is not None])
    
    if all_scores:
        print(f"\nOverall Statistics:")
        print(f"Mean Dice: {np.mean(all_scores):.3f} ± {np.std(all_scores):.3f}")
        print(f"Total measurements: {len(all_scores)}")
        print(f"Total slices analyzed: {len(percentiles)}")
    
    print("="*70)
    
    return slice_metrics

# Usage examples:

# Example 1: Show bounding boxes only on specific percentiles
slice_metrics = visualize_percentile_slices(
    img_3D, gt_3D, pred_3D, 
    labels=labels, 
    percentiles=[10, 20, 25, 35, 50, 65, 75, 80, 90],  # All percentiles to show
    bbox_percentiles=[25, 50, 75],  # Only show bboxes on these percentiles
    key_slice_idx=33,  # Reference slice for bbox extraction
    CLASS_COLORS=CLASS_COLORS,
    figsize_per_image=(5, 3)
)

# Example 2: Show bounding boxes on all percentiles
# slice_metrics = visualize_percentile_slices(
#     img_3D, gt_3D, pred_3D, 
#     labels=labels, 
#     percentiles=[20, 35, 50, 65, 80],
#     bbox_percentiles=[20, 35, 50, 65, 80],  # Same as percentiles - show bboxes on all
#     key_slice_idx=33,
#     CLASS_COLORS=CLASS_COLORS,
#     figsize_per_image=(6, 4)
# )

# Example 3: No bounding boxes (original behavior)
# slice_metrics = visualize_percentile_slices(
#     img_3D, gt_3D, pred_3D, 
#     labels=labels, 
#     percentiles=[10, 25, 50, 75, 90],
#     bbox_percentiles=[],  # Empty list - no bboxes
#     CLASS_COLORS=CLASS_COLORS,
#     figsize_per_image=(5, 3)
# )

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import cv2
from pathlib import Path

def mask2D_to_bbox(mask_2d):
    """Extract bounding box from 2D mask"""
    if np.sum(mask_2d) == 0:
        return None
    rows, cols = np.where(mask_2d > 0)
    y_min, y_max = rows.min(), rows.max()
    x_min, x_max = cols.min(), cols.max()
    return [x_min, y_min, x_max, y_max]

def get_class_bboxes_2d(mask_slice, class_ids):
    """
    Extract bounding boxes for each class in a 2D slice
    
    Args:
        mask_slice: 2D numpy array with class labels
        class_ids: list of class IDs to extract bboxes for
    
    Returns:
        dict: {class_id: bbox} where bbox is [x_min, y_min, x_max, y_max] or None if class not present
    """
    bboxes = {}
    for class_id in class_ids:
        class_mask = (mask_slice == class_id)
        bbox = mask2D_to_bbox(class_mask)
        # bbox= custom # Your hardcoded bbox
        

        bboxes[class_id] = bbox

    return bboxes



def visualize_percentile_slices(img_3D, gt_3D, pred_3D, labels,key_slice_idx, percentiles=[10, 25, 50, 75, 90], CLASS_COLORS=None):
    """
    Visualize prediction performance across multiple percentile slices
    
    Args:
        img_3D: 3D image array [D, H, W]
        gt_3D: 3D ground truth array [D, H, W]  
        pred_3D: 3D prediction array [D, H, W]
        labels: list of class labels to analyze
        percentiles: list of percentiles to visualize (default: [10, 25, 50, 75, 90])
        CLASS_COLORS: dictionary mapping class_id to RGB colors
    """
    
    if CLASS_COLORS is None:
     CLASS_COLORS = {
        0: (0, 0, 0),         # background
        1: (255, 0, 0),       # red
        2: (0, 255, 0),       # green
        3: (0, 0, 255),       # blue
        4: (255, 255, 0),     # yellow
        5: (255, 0, 255),     # magenta
        6: (0, 255, 255),     # cyan
        7: (255, 165, 0),     # orange ← updated
    }

    
    D = img_3D.shape[0]
    n_percentiles = len(percentiles)
    
    # Calculate slice indices for each percentile
    slice_indices = []
    for p in percentiles:
        slice_idx = int(D * p / 100)
        slice_indices.append(min(slice_idx, D-1))  # Ensure we don't exceed bounds
    
    # Create figure with subplots
    fig, axes = plt.subplots(4, n_percentiles, figsize=(4*n_percentiles, 16))
    
    if n_percentiles == 1:
        axes = axes.reshape(-1, 1)
    
    # Store metrics for summary
    slice_metrics = {}
    
    for col, (percentile, slice_idx) in enumerate(zip(percentiles, slice_indices)):
        img_slice = img_3D[slice_idx]
        gt_slice = gt_3D[slice_idx]
        pred_slice = pred_3D[slice_idx]
        key_slice=gt_3D[key_slice_idx]
        
        # Contrast stretching for image
        vmin, vmax = np.percentile(img_slice, [1, 99])
        img_norm = np.clip((img_slice - vmin) / (vmax - vmin + 1e-8), 0, 1)
        img_rgb = (np.stack([img_norm]*3, axis=-1) * 255).astype(np.uint8)
        
        # Convert masks to RGB
        gt_rgb = mask_ids_to_rgb(gt_slice, CLASS_COLORS)
        pred_rgb = mask_ids_to_rgb(pred_slice, CLASS_COLORS)
        
        # Create overlay
        overlay = enhance_overlay(img_rgb, pred_rgb, alpha=0.5, beta=0.1)

        
        # Get bounding boxes for this slice
        gt_bboxes = get_class_bboxes_2d(key_slice, labels)
        pred_bboxes = get_class_bboxes_2d(pred_slice, labels)
        img_with_boxes = draw_multiple_bboxes(img_rgb, gt_bboxes, CLASS_COLORS, thickness=1)
        gt_img_with_boxes = draw_multiple_bboxes(gt_rgb, gt_bboxes, CLASS_COLORS, thickness=1)

        
        # Plot images
        axes[0, col].imshow(img_with_boxes)
        axes[0, col].set_title(f'{percentile}th', fontsize=10)
        axes[0, col].axis('off')
        
        axes[1, col].imshow(gt_img_with_boxes)
        axes[1, col].set_title(f'GT Classes', fontsize=10)
        axes[1, col].axis('off')
        
        axes[2, col].imshow(pred_rgb)
        axes[2, col].set_title(f'Pred Classes', fontsize=10)
        axes[2, col].axis('off')
        
        axes[3, col].imshow(overlay)
        axes[3, col].set_title('Overlay', fontsize=10)
        axes[3, col].axis('off')
        
        # Calculate metrics for this slice
        slice_gt_bool = np.isin(gt_slice, labels)
        slice_pred_bool = np.isin(pred_slice, labels)
        
        if np.sum(slice_gt_bool) > 0:  # Only calculate if GT has annotations
            slice_dice = dice_score(slice_pred_bool, slice_gt_bool)
            slice_iou = iou_score(slice_pred_bool, slice_gt_bool)
        else:
            slice_dice = 0.0
            slice_iou = 0.0
        
        slice_metrics[percentile] = {
            'slice_idx': slice_idx,
            'dice': slice_dice,
            'iou': slice_iou,
            'gt_classes': list(np.unique(gt_slice)),
            'pred_classes': list(np.unique(pred_slice)),
            'gt_pixel_count': np.sum(slice_gt_bool),
            'pred_pixel_count': np.sum(slice_pred_bool)
        }
    
    plt.tight_layout()
    plt.show()

    return slice_metrics

# Use your existing data
print("Visualizing prediction performance across multiple percentile slices...")

# You can customize which percentiles to visualize
# percentiles_to_show = [10, 15, 50, 75, 90]  # or [20, 40, 60, 80] or any other values
# percentiles_to_show = [25, 50,65, 75,]  # More extreme percentiles
percentiles_to_show = [20, 35, 50, 65, 80]  # Around your key slice

slice_metrics = visualize_percentile_slices(
    img_3D, gt_3D, pred_3D, 
    labels=labels, 
    key_slice_idx=33,
    percentiles=percentiles_to_show,
    CLASS_COLORS=CLASS_COLORS
)
